<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-09-ai-agents/lesson-9.3-langgraph-deep-dive/notebooks/exercises-9_3.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 9.3: LangGraph Deep Dive

*Module 9 · AI Agents & Autonomous Systems*

> 9.1 coded agents from scratch. 9.2 compared architectures. LangGraph makes those patterns production-ready: typed state, conditional routing, checkpointed persistence, streaming, and human-in-the-loop — all as a graph of nodes and edges. This lesson builds three real LangGraph agents, from minimal to production-grade.

`StateGraph` · `Conditional Edges` · `Persistence` · `Streaming` · `75 min`

---

## About this notebook

This notebook contains the **3 runnable code examples** from the published lesson `lesson-9.3.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — Minimal LangGraph Agent — Nodes, Edges, State — `01_minimal_langgraph.py`
2. Step 2 — Conditional Routing — Branch by Topic — `02_conditional_routing.py`
3. Step 3 — Persistence — Save and Resume Agent State — `03_persistence.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q langchain langgraph


In [ ]:
# Load API keys from the environment.
# Before running the lesson cells, export each key in your shell, e.g.:
#   export GOOGLE_API_KEY=sk-...
# (Or replace the placeholder below with your real key for a quick local test.)
import os
os.environ.setdefault("GOOGLE_API_KEY", "YOUR_GOOGLE_API_KEY_HERE")


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · Minimal LangGraph Agent — Nodes, Edges, State

The simplest possible LangGraph: agent node + tool node + conditional edge.

**`01_minimal_langgraph.py`** · _Block 1 of 3_

MINIMAL LANGGRAPH AGENT — pip install langgraph langchain-google-genai


In [ ]:
# MINIMAL LANGGRAPH AGENT
# pip install langgraph langchain-google-genai
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import operator, os

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY", "your-key")

# ── 1. Define State ──
class AgentState(TypedDict):
    messages: Annotated[list, operator.add]  # append-only message list

# ── 2. Define Tools ──
@tool
def search_courses(topic: str) -> str:
    """Search Netsetos courses by topic."""
    db = {"genai":"GenAI: 14999 INR, 146hrs", "python":"Python: 9999 INR, 80hrs"}
    return db.get(topic.lower(), "Not found")

@tool
def calculate(expression: str) -> str:
    """Evaluate a math expression like '14999 * 1.18'."""
    return str(round(eval(expression, {"__builtins__":{}}), 2))

tools = [search_courses, calculate]

# ── 3. Define Nodes ──
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash").bind_tools(tools)

def agent_node(state: AgentState):
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

tool_node = ToolNode(tools)

# ── 4. Define Edges (routing logic) ──
def should_continue(state: AgentState):
    last = state["messages"][-1]
    if hasattr(last, "tool_calls") and last.tool_calls:
        return "tools"     # LLM wants to call a tool
    return "end"          # LLM has final answer

# ── 5. Build Graph ──
graph = StateGraph(AgentState)
graph.add_node("agent", agent_node)
graph.add_node("tools", tool_node)
graph.set_entry_point("agent")
graph.add_conditional_edges("agent", should_continue, {"tools":"tools", "end":END})
graph.add_edge("tools", "agent")  # after tools, go back to agent

app = graph.compile()

# ── 6. Run ──
result = app.invoke({"messages": [HumanMessage("What is the GenAI course price with 18% GST?")]})
print("LangGraph Agent:")
print(f"  Answer: {result['messages'][-1].content[:120]}")
print(f"  Messages: {len(result['messages'])} total")


> **What just happened?** Five components = a complete LangGraph agent: (1) State : TypedDict with append-only messages. (2) Nodes : agent_node (LLM) and tool_node (execute). (3) Conditional edge : should_continue checks for tool_calls. (4) Loop : tools → agent edge creates the cycle. (5) Compile & invoke . This replaces the while-loop from 9.1 with a graph that can be checkpointed, streamed, and visualized.


### Step 2 · Conditional Routing — Branch by Topic

Route different queries to different nodes: support, billing, technical.

**`02_conditional_routing.py`** · _Block 2 of 3_

CONDITIONAL ROUTING — Different paths for different queries


In [ ]:
# CONDITIONAL ROUTING — Different paths for different queries
from typing import TypedDict, Annotated, Literal
from langgraph.graph import StateGraph, END
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage
import operator, os

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY", "your-key")

class State(TypedDict):
    messages: Annotated[list, operator.add]
    route: str

llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash")

# ── Router node: classify the query ──
def router(state: State):
    query = state["messages"][-1].content.lower()
    if any(w in query for w in ["refund","price","cost","pay","emi"]):
        return {"route": "billing"}
    elif any(w in query for w in ["python","prerequisite","curriculum","module"]):
        return {"route": "technical"}
    return {"route": "general"}

# ── Specialist nodes ──
def billing_node(state: State):
    resp = llm.invoke([HumanMessage(f"As a billing specialist, answer: {state['messages'][-1].content}\nRefund: full 7d, 50% 30d. GenAI: 14999. EMI available.")])
    return {"messages": [resp]}

def technical_node(state: State):
    resp = llm.invoke([HumanMessage(f"As a technical advisor, answer: {state['messages'][-1].content}\nPrereqs: Python, HS math. 14 modules, 146 hrs.")])
    return {"messages": [resp]}

def general_node(state: State):
    resp = llm.invoke([HumanMessage(f"As a Netsetos assistant, answer: {state['messages'][-1].content}")])
    return {"messages": [resp]}

# ── Build graph with conditional routing ──
def route_decision(state: State) -> Literal["billing","technical","general"]:
    return state["route"]

graph = StateGraph(State)
graph.add_node("router", router)
graph.add_node("billing", billing_node)
graph.add_node("technical", technical_node)
graph.add_node("general", general_node)

graph.set_entry_point("router")
graph.add_conditional_edges("router", route_decision,
    {"billing":"billing", "technical":"technical", "general":"general"})
graph.add_edge("billing", END)
graph.add_edge("technical", END)
graph.add_edge("general", END)

app = graph.compile()

print("Conditional Routing:\n")
for q in ["Can I get a refund?", "What Python version is needed?", "Hello!"]:
    r = app.invoke({"messages":[HumanMessage(q)], "route":""})
    print(f"  Q: {q}")
    print(f"  Route: {r['route']} → {r['messages'][-1].content[:80]}\n")


### Step 3 · Persistence — Save and Resume Agent State

Checkpoint every step to SQLite. Resume after crash, review history, time-travel debug.

**`03_persistence.py`** · _Block 3 of 3_

LANGGRAPH PERSISTENCE — SQLite checkpointing — pip install langgraph-checkpoint-sqlite


In [ ]:
# LANGGRAPH PERSISTENCE — SQLite checkpointing
# pip install langgraph-checkpoint-sqlite
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.sqlite import SqliteSaver
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import operator, os

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY", "your-key")

class State(TypedDict):
    messages: Annotated[list, operator.add]

@tool
def search(query: str) -> str:
    """Search knowledge base."""
    return {"genai":"14999 INR"}.get(query.lower(), "Not found")

tools = [search]
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash").bind_tools(tools)

def agent(state): return {"messages": [llm.invoke(state["messages"])]}
def should_continue(state):
    last = state["messages"][-1]
    return "tools" if hasattr(last,"tool_calls") and last.tool_calls else "end"

g = StateGraph(State)
g.add_node("agent", agent)
g.add_node("tools", ToolNode(tools))
g.set_entry_point("agent")
g.add_conditional_edges("agent", should_continue, {"tools":"tools", "end":END})
g.add_edge("tools", "agent")

# ── SQLite checkpointer ──
with SqliteSaver.from_conn_string(":memory:") as checkpointer:
    app = g.compile(checkpointer=checkpointer)

    # Thread = conversation session
    config = {"configurable": {"thread_id": "user-123"}}

    # Turn 1
    r1 = app.invoke({"messages":[HumanMessage("GenAI course price?")]}, config)
    print("Turn 1:", r1["messages"][-1].content[:60])

    # Turn 2: continues same thread (has memory!)
    r2 = app.invoke({"messages":[HumanMessage("With 18% GST?")]}, config)
    print("Turn 2:", r2["messages"][-1].content[:60])

    # View checkpoint history
    for cp in checkpointer.list(config):
        print(f"  Checkpoint: {cp.metadata.get('step',0)} msgs")

print("\nPersistence:")
print("  SQLite: SqliteSaver.from_conn_string('agent.db')")
print("  Postgres: PostgresSaver (production)")
print("  Every node execution is checkpointed")
print("  Resume: same thread_id = continue conversation")


> **What just happened?** SqliteSaver checkpoints every node execution. Turn 2 (“With 18% GST?”) understood “it” = GenAI course because the full message history was persisted. Change :memory: to agent.db and state survives server restarts. For production, use PostgresSaver. thread_id = session ID per user.


---

## Wrap-up

You've walked through all 3 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
